![Silver LOGO](https://carboncredits.b-cdn.net/wp-content/uploads/2024/04/shutterstock_1447266653.jpg)



# CAMADA SILVER

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import *
# from pyspark.sql.functions import col
import pytz
# from pyspark.sql.functions import lit
from pyspark.sql import Window
# from pyspark.sql import F
import pyspark.sql.functions as F
from datetime import datetime , timedelta

import requests
import json



## PARÂMETROS

In [0]:
try:
  time_file = datetime.now(pytz.timezone('America/Sao_Paulo')).strftime('%Y%m%d_%H%M%S')
  # DeltaTable.isDeltaTable(spark, silver_delta_path)

  #### Nome  e caminho onde será feita a escrita da da tabela Delta  ##########
  catalog = 'databricks_proj'
  db_bronze = 'bronze'
  db = 'silver'
  table = 'ibge_news'
  silver_delta_table = f'{db}.{table}'
  silver_delta_path = '/Volumes/workspace/default/silver/' 
  DT_END = datetime.now().date()
  # DT_START = (DT_END - timedelta(days = 7))

  #### spark.catalog.tableExists PARA VERIFICAR SE EXISTE A TABELA NO CATALOGO  #####
  if spark.catalog.tableExists(f'{catalog}.{db}.{table}'):
    print('delta table existe\n')
    df_silver = spark.sql(f"select * from {catalog}.{db}.{table} ")
    if len(df_silver.take(1)) >0:
      print('Tabela Silver ja existe\n')
      DT_START = spark.sql(f"select cast(trunc(to_date(max(left(dateIngestion,10) ),'yyyy-MM-dd'),'MM') as string) from {catalog}.{db}.{table} ").collect()[0][0] ##trunc DO Mês
    else:
      print('Tabela Silver ja existe. Mas ainda nao tem dados')
      DT_START = spark.sql(f"select cast(trunc(to_date(min(left(DTPROC,8 ) ),'yyyyMMdd'),'MM') as string) from {catalog}.{db_bronze}.{table} ").collect()[0][0]

  else: ## Se nao existir tabela silver, pegar menor data da tabela bronze
    DT_START = spark.sql(f"select cast(trunc(to_date(min(left(DTPROC,8 ) ),'yyyyMMdd'),'MM') as string) from {catalog}.{db_bronze}.{table} ").collect()[0][0]

  ### Sistema de origem ####
  source_delta_table = f'{catalog}.bronze.ibge_news'
  
  print("##----------------------------##")
  print(f"Data e Horário de execusão             ===>  {time_file} \n")
  print(f"DT_START                               ===>  {DT_START} ")
  print(f"DT_END                                 ===>  {DT_END}")
  print(f"Catalago                                ===>  {catalog}")
  print(f"Banco de Dados                         ===>  {db}")
  print(f"Nome da Tabela                         ===>  {table}")
  print(f"Tabela silver a ser criada             ===>  {silver_delta_table}")
  print(f"Caminho da tabela silver a ser criada  ===>  {silver_delta_path} \n")
  print("##----------------------------##")
  print(f"Tabela de Origem                     ===> {source_delta_table}")

except Exception as e:
  print(e)

In [0]:
class  SilverIngestion:
  def __init__(self,catalog,db,source_delta_table,silver_delta_table,silver_delta_path,dtstart,dtend):
    self.catalog = catalog
    self.db = db
    self.source_delta_table = source_delta_table
    self.silver_delta_table = silver_delta_table
    self.silver_delta_path = silver_delta_path
    self.dtstart = dtstart
    self.dtend = dtend
    self.dt = str(dtend).replace('-','')[0:6]

  def create_structured_delta_schema(self): ## Ajustar ceate Table com Volumes

    if not DeltaTable.isDeltaTable(spark,self.silver_delta_path):
      print('Criando estrutura tabela Delta....\n')
      schema = StructType([

                      StructField('referenceMonthDate',DateType(),True,metadata={"comment": "Data do Mes de referencia"}),
                      StructField('id',LongType(),True,metadata={"comment": "Identificador único da notícia"}),
                      StructField('newsHighlight', StringType(),True,metadata={"comment": "Destaque"}),
                      StructField('editorials', StringType(),True,metadata={"comment": "Editorial"}),
                      StructField('images', StringType(),True,metadata={"comment": "Descrição das imagens"}),
                      StructField('publicationDate', TimestampType(),True,metadata={"comment": "Data da publicaçao"}),
                      StructField('introduction', StringType(),True,metadata={"comment": "Introduçao da Noticia"}),
                      StructField('link', StringType(),True,metadata={"comment": "Link da Noticia"}),
                      StructField('product_id', StringType(),True,metadata={"comment": "ID do produto"}),
                      StructField('products', StringType(),True,metadata={"comment": "Produto"}),
                      StructField('relatedProducts', StringType(),True,metadata={"comment": "Produtos relacionados"}),
                      StructField('type', StringType(),True,metadata={"comment": "Tipo "}),
                      StructField('title', StringType(),True,metadata={"comment": "Título da Noticia"}),
                      StructField('dateIngestion', TimestampType(),True,metadata={"comment": "Data de Ingestão"}),
                      StructField('dt', StringType(),True,metadata={"comment": "Data para partiçao"})
                            ])
      ### Criando estrutura tabela Delta ###
      df =  spark.createDataFrame(data= [],schema=schema)
      print(f'Criando estrutura da tabela delta no caminho .. {self.silver_delta_path}')
      df.write.format('delta').partitionBy('dt').save(f'{self.silver_delta_path}')

  def transform(self):
    try:  
      self.create_structured_delta_schema()
      columns_json ={"id": "id","newsHighlight": "destaque","editorials": "editorias","images": "imagens","introduction": "introducao","link": "link","product_id":"produto_id","products": "produtos","relatedProducts": "produtos_relacionados","type": "tipo","title": "titulo"}
      df_silver_columns = spark.sql(f"""SELECT * FROM delta.`{self.silver_delta_path}` """).columns
      columns_silver = []
      for columns in df_silver_columns:
        # if not  columns in ['referenceMonthDate','publicationDate','dateIngestion','dt']:
        col_string = F.col(columns_json[columns]).alias(columns)
        columns_silver.append(col_string)
      # É feito um filtro de data por data inicial(a data mais recente da tabela bronze ou data mais antiga da tabela silver)
      # e data final (correpondente pela data corrente)
      df = spark.table(self.source_delta_table).filter(F.to_date(F.substring(F.col('DTPROC'),1,8 ),'yyyyMMdd').between(f"{self.dtstart}",f"{self.dtend}"))
      ### função de janela para efetuar a deduplicaçao dos dados   #####
      row_numer_experssion = Window.partitionBy(F.col('id')).orderBy(F.col('DTPROC').desc())

      df_stage = (df
                      .withColumn('referenceMonthDate',F.trunc(F.to_date(F.col('data_publicacao'),'dd/MM/yyyy HH:mm:ss').cast('date'),'MM'))
                      .withColumn('publicationDate', F.to_timestamp(F.col('data_publicacao'),'dd/MM/yyyy HH:mm:ss').cast('timestamp'))
                      .withColumn('dateIngestion', F.lit(datetime.now() - timedelta(hours = 3)).cast('timestamp')) 
                      .withColumn('dt', F.lit(f"{self.dt}"))
                      .withColumn('rownumber_wdw', F.row_number().over(row_numer_experssion)).filter(F.col("rownumber_wdw") == 1)
                  ).drop('rownumber_wdw')

      df_final = (df_stage.select('referenceMonthDate','publicationDate',*columns_silver,'dateIngestion','dt') )
      return df_final

    except Exception as error:    
      raise ValueError(f"{error}")
    
  def save_silver(self):
    try:
      df_final = self.transform()

      print('Inicio gravação tabela delta...\n')
      (DeltaTable.forPath(spark, self.silver_delta_path).alias("old")
       .merge(df_final.alias("new"),"old.id = new.id")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll().execute()
       )
      ### Criando Data base  ###
      sql = f""" CREATE DATABASE IF NOT EXISTS {self.catalog }.{db} """
      spark.sql(sql)
      ### Criando tabela de metadadados db silver  ###
      sql =  f""" DROP TABLE IF EXISTS {self.catalog }.{self.silver_delta_table}"""
      spark.sql(sql)
      # print(sql)

      sql =  f""" CREATE OR REPLACE TABLE {self.catalog }.{self.silver_delta_table} AS
                  SELECT * FROM delta.`{self.silver_delta_path}` """
      spark.sql(sql)
      # print(sql,'\n') 
      print(f'Tabela {self.catalog }.{self.silver_delta_table}  criada com sucesso !!!\n')
      print('Gravação finalizada com sucesso!!!\n')

      sql_opt = f"""   OPTIMIZE {self.catalog }.{self.silver_delta_table} ZORDER BY (id);
                                      """
      spark.sql(sql_opt)
      print(f'Otimizando a tabela ===> {self.catalog }.{self.silver_delta_table}') 

    except Exception as error: 
      raise ValueError(f"{error}")


# FAzer Adicionar ao JOB


In [0]:
silver_class = SilverIngestion(catalog,db,source_delta_table,silver_delta_table,silver_delta_path,DT_START,DT_END)
silver_class.save_silver()
  

# Demora para o cluster entender que DeltaTable nao existe
- Tabbela silver criada com AS SELECT   e volume para arquivos criados criado tbm

In [0]:
if not DeltaTable.isDeltaTable(spark,'/Volumes/workspace/default/silver/_delta_log'):
  print(DeltaTable.isDeltaTable(spark,'/Volumes/workspace/default/silver/_delta_log'))

In [0]:
#### Colocar OPTIMIZE no final da gravaçao da tabela

# OPTIMIZE silver.ibge_news

MONTAR E SELECIONAR DATAFRAME COM FUNÇAO DE JANELA E TRATAMENTO DE ALGUNS CAMPOS. (MUDAR NOMES PAFRA INGLES)

In [0]:
# display(dbutils.fs.ls('dbfs:/mnt/silver/'))
# display(dbutils.fs.ls('dbfs:/mnt/historic/bronze/202410/ibgeapipage_1_to_10_20241019_110702/'))

## apagar pastas
dbutils.fs.rm('/Volumes/workspace/default/silver/_delta_log/',True)
# dbutils.fs.rm('dbfs:/mnt/bronze/',True)
# dbutils.fs.rm('dbfs:/mnt/historic/bronze/',True)
# dbutils.fs.rm(silver_delta_path,True)